In [3]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve()
while project_root != project_root.parent:
    if (project_root / 'src').exists():
        break
    project_root = project_root.parent
else:
    raise RuntimeError('Unable to locate project root containing src/')

src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))


# RPFR Table 4 Replication

Reproduce the HD/H$_2$, DF/HF, and C$^{18}$O/C$^{16}$O contributions reported in Table 4 of Richet et al. (1977) using the processed diatom constants.

In [4]:
from pathlib import Path

import pandas as pd

from richet_rpfr import (
    load_molecular_constants_from_excel,
    compute_and_finalize_with_temp,
    create_summary_df,
)

raw_excel = project_root / 'data' / 'processed' / 'spreadsheets' / 'Richet - RPFR & mol constans - diatoms.xlsx'
constants = load_molecular_constants_from_excel(raw_excel, sheet_name='Diatoms (Table 5)')

NSI_H2 = constants['1H1H']
SSI_H2 = constants['1H2H']
NSI_HF = constants['1H19F']
SSI_HF = constants['2H19F']
NSI_CO = constants['12C16O']
SSI_CO = constants['12C18O']

T_0C = 273.15

paper_HD_H2 = {
    'Translational': 1.83561,
    'ZPE_G0': 1.00910,
    'ZPE_harmonic': 4.71546,
    'ZPE_anharmonic': 0.96079,
    'ZPE_total': 4.57179,
    'Rotational Diatomic': 2.59246,
    'Total': 21.75600,
}
paper_DF_HF = {
    'Translational': 1.07638,
    'ZPE_G0': 1.01192,
    'ZPE_harmonic': 20.1428,
    'ZPE_anharmonic': 0.94356,
    'ZPE_total': 19.2325,
    'Rotational Diatomic': 1.86165,
    'Total': 38.5389,
}
paper_C18O_C16O = {
    'Translational': 1.10929,
    'ZPE_G0': 1.00005,
    'ZPE_harmonic': 1.14804,
    'ZPE_anharmonic': 0.99917,
    'ZPE_total': 1.14714,
    'Rotational Diatomic': 1.04985,
    'Total': 1.33595,
}

print()

final_HD_H2 = compute_and_finalize_with_temp(T_0C, NSI_H2, SSI_H2, paper_HD_H2)
final_DF_HF = compute_and_finalize_with_temp(T_0C, NSI_HF, SSI_HF, paper_DF_HF)
final_C18O_C16O = compute_and_finalize_with_temp(T_0C, NSI_CO, SSI_CO, paper_C18O_C16O)

summary = create_summary_df(
    final_HD_H2, paper_HD_H2,
    final_DF_HF, paper_DF_HF,
    final_C18O_C16O, paper_C18O_C16O,
)
pd.set_option('display.max_columns', None)
summary


Contribution     HD/H2                               DF/HF  \
                                 Paper   Computed Difference (‰)     Paper   
0              Translational   1.83561   1.835706       0.052483   1.07638   
1                     ZPE_G0   1.00910   1.009100       0.000000   1.01192   
2               ZPE_harmonic   4.71546   4.715118      -0.072545  20.14280   
3             ZPE_anharmonic   0.96079   0.960794       0.004421   0.94356   
4                  ZPE_total   4.57179   4.571484      -0.067037  19.23250   
5     Excited state harmonic       NaN   1.000000            NaN       NaN   
6   Excited state anharmonic       NaN   1.000000            NaN       NaN   
7          Rotational linear       NaN   2.587940            NaN       NaN   
8        Rotational Diatomic   2.59246   2.587775      -1.807324   1.86165   
9     Rotational-vibrational       NaN   1.000000            NaN       NaN   
10                     Total  21.75600  21.716349      -1.822539  38.53890   

                             C18O/C16O                           
     Computed Difference (‰)     Paper  Computed Difference (‰)  
0    1.076388       0.007589   1.10929  1.109290      -0.000428  
1    1.011920       0.000000   1.00005  1.000050       0.000000  
2   20.140387      -0.119817   1.14804  1.148032      -0.006776  
3    0.943558      -0.002240   0.99917  0.999164      -0.005847  
4   19.230144      -0.122515   1.14714  1.147130      -0.008717  
5    1.000000            NaN       NaN  1.000003            NaN  
6    1.000000            NaN       NaN  1.000000            NaN  
7    1.861662            NaN       NaN  1.049839            NaN  
8    1.861655       0.002845   1.04985  1.049839      -0.010902  
9    1.000000            NaN       NaN  1.000000            NaN  
10  38.534590      -0.111822   1.33595  1.335923      -0.020271